In [2]:
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u

def generate_apt_polygon(coords, sky_padding_deg=0.35):
    """
    Calculates a bounding box polygon for a set of ICRS coordinates.
    Default padding is 0.35 degrees (roughly half a Roman WFI tile).
    """
    # Extract RA and Dec in degrees
    ras = [coord.ra.degree for coord in coords]
    decs = [coord.dec.degree for coord in coords]
    
    min_ra, max_ra = min(ras), max(ras)
    min_dec, max_dec = min(decs), max(decs)
    
    # Adjust RA padding for spherical distortion at this declination
    mean_dec = np.mean([min_dec, max_dec])
    ra_pad = sky_padding_deg / np.cos(np.radians(mean_dec))
    
    # Define the 4 corners of the bounding box
    corners = [
        SkyCoord(ra=(min_ra - ra_pad)*u.deg, dec=(min_dec - sky_padding_deg)*u.deg, frame='icrs'),
        SkyCoord(ra=(max_ra + ra_pad)*u.deg, dec=(min_dec - sky_padding_deg)*u.deg, frame='icrs'),
        SkyCoord(ra=(max_ra + ra_pad)*u.deg, dec=(max_dec + sky_padding_deg)*u.deg, frame='icrs'),
        SkyCoord(ra=(min_ra - ra_pad)*u.deg, dec=(max_dec + sky_padding_deg)*u.deg, frame='icrs')
    ]
    
    return corners

# Initialize our specific targets
# KOI-7179 Coordinates
koi_7179 = SkyCoord(ra=297.37428*u.deg, dec=46.05482*u.deg, frame='icrs')
# KOI-8174 Coordinates
koi_8174 = SkyCoord(ra=286.94946*u.deg, dec=45.14214*u.deg, frame='icrs')

targets = [koi_7179, koi_8174]

# Generate polygon (0.35 degree padding)
polygon_corners = generate_apt_polygon(targets, sky_padding_deg=0.35)

print("ICRS Perimeter Coordinates for APT Region Planner:\n")
for i, corner in enumerate(polygon_corners, 1):
    # Output in both sexagesimal and decimal formats for easy APT entry
    ra_str = corner.ra.to_string(unit=u.hour, sep=':', precision=2)
    dec_str = corner.dec.to_string(unit=u.degree, sep=':', precision=2)
    
    print(f"Vertex {i}: RA {ra_str}, Dec {dec_str}")
    print(f"          (Decimal: {corner.ra.degree:.5f}, {corner.dec.degree:.5f})\n")

ICRS Perimeter Coordinates for APT Region Planner:

Vertex 1: RA 19:05:47.82, Dec 44:47:31.70
          (Decimal: 286.44923, 44.79214)

Vertex 2: RA 19:51:29.88, Dec 44:47:31.70
          (Decimal: 297.87451, 44.79214)

Vertex 3: RA 19:51:29.88, Dec 46:24:17.35
          (Decimal: 297.87451, 46.40482)

Vertex 4: RA 19:05:47.82, Dec 46:24:17.35
          (Decimal: 286.44923, 46.40482)



In [3]:
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u

def generate_tight_polygon(target_coord, box_size_deg=0.35):
    """
    Calculates a tight bounding box polygon around a single target.
    A box size of 0.35 deg creates a roughly 0.7 deg wide box, 
    similar to a single Roman WFI tile.
    """
    ra = target_coord.ra.degree
    dec = target_coord.dec.degree
    
    # Adjust RA padding for spherical distortion at this declination
    ra_pad = box_size_deg / np.cos(np.radians(dec))
    
    corners = [
        SkyCoord(ra=(ra - ra_pad)*u.deg, dec=(dec - box_size_deg)*u.deg, frame='icrs'),
        SkyCoord(ra=(ra + ra_pad)*u.deg, dec=(dec - box_size_deg)*u.deg, frame='icrs'),
        SkyCoord(ra=(ra + ra_pad)*u.deg, dec=(dec + box_size_deg)*u.deg, frame='icrs'),
        SkyCoord(ra=(ra - ra_pad)*u.deg, dec=(dec + box_size_deg)*u.deg, frame='icrs')
    ]
    return corners

# Initialize our specific targets
koi_7179 = SkyCoord(ra=297.37428*u.deg, dec=46.05482*u.deg, frame='icrs')
koi_8174 = SkyCoord(ra=286.94946*u.deg, dec=45.14214*u.deg, frame='icrs')

# Generate separate polygons
poly_7179 = generate_tight_polygon(koi_7179)
poly_8174 = generate_tight_polygon(koi_8174)

def print_vertices(target_name, corners):
    print(f"--- ICRS Perimeter for {target_name} ---")
    for i, corner in enumerate(corners, 1):
        ra_str = corner.ra.to_string(unit=u.hour, sep=':', precision=2)
        dec_str = corner.dec.to_string(unit=u.degree, sep=':', precision=2)
        print(f"Vertex {i}: RA {ra_str}, Dec {dec_str}")
    print("\n")

print_vertices("KOI-7179 Region", poly_7179)
print_vertices("KOI-8174 Region", poly_8174)

--- ICRS Perimeter for KOI-7179 Region ---
Vertex 1: RA 19:47:28.78, Dec 45:42:17.35
Vertex 2: RA 19:51:30.87, Dec 45:42:17.35
Vertex 3: RA 19:51:30.87, Dec 46:24:17.35
Vertex 4: RA 19:47:28.78, Dec 46:24:17.35


--- ICRS Perimeter for KOI-8174 Region ---
Vertex 1: RA 19:05:48.78, Dec 44:47:31.70
Vertex 2: RA 19:09:46.96, Dec 44:47:31.70
Vertex 3: RA 19:09:46.96, Dec 45:29:31.70
Vertex 4: RA 19:05:48.78, Dec 45:29:31.70




In [4]:
from astroquery.simbad import Simbad

# Create a custom Simbad query object to include proper motion fields
custom_simbad = Simbad()
custom_simbad.add_votable_fields('pmra', 'pmdec')

targets = ["KOI-7179", "KOI-8174"]

print("Fetching target data from SIMBAD...\n")

for target in targets:
    result = custom_simbad.query_object(target)
    
    if result is not None:
        # SIMBAD returns RA and Dec as sexagesimal strings by default
        # Keys updated to lowercase for newer astroquery versions
        ra = result['ra'][0]
        dec = result['dec'][0]
        
        # Proper motion in milliarcseconds per year (mas/yr)
        pmra = result['pmra'].data[0] 
        pmdec = result['pmdec'].data[0]
        
        # SIMBAD's standard coordinate reference frame/epoch
        epoch = "J2000.0"
        
        print(f"--- {target} ---")
        print(f"RA:        {ra}")
        print(f"Dec:       {dec}")
        # Formatting to handle potential missing PM data gracefully
        print(f"PM RA:     {pmra if not str(pmra).startswith('--') else '0.0'} mas/yr")
        print(f"PM Dec:    {pmdec if not str(pmdec).startswith('--') else '0.0'} mas/yr")
        print(f"Epoch:     {epoch}\n")
    else:
        print(f"Could not find data for {target} in SIMBAD.\n")

Fetching target data from SIMBAD...

--- KOI-7179 ---
RA:        297.37428203626
Dec:       46.054821593839996
PM RA:     -4.763 mas/yr
PM Dec:    -16.699 mas/yr
Epoch:     J2000.0

--- KOI-8174 ---
RA:        286.94946367209
Dec:       45.142091900980006
PM RA:     -15.863 mas/yr
PM Dec:    -77.249 mas/yr
Epoch:     J2000.0



In [5]:
def calculate_roman_pass_plan(duration_hours, sci_time_sec, exec_time_sec, num_targets=1):
    """
    Calculates the APT parameters for a continuous stare Pass Plan.
    """
    # Convert duration to seconds
    total_time_sec = duration_hours * 3600
    
    # Calculate observations (exposures) that fit into the block
    # We use floor division to ensure we don't exceed the 4-hour window
    obs_per_segment = int(total_time_sec // exec_time_sec)
    
    # Calculate actual times and efficiency
    actual_exec_time = obs_per_segment * exec_time_sec
    actual_sci_time = obs_per_segment * sci_time_sec
    efficiency = (actual_sci_time / actual_exec_time) * 100
    
    # APT structural definitions for a continuous stare on fixed targets:
    # A single continuous block on one target (Segment) is 1 Visit.
    visits_per_segment = 1 
    
    # Total visits in the Pass Plan depends on how many targets are included.
    # If KOI-7179 and KOI-8174 are in the same Pass Plan, it's 2 Visits.
    visits_per_pass = visits_per_segment * num_targets

    print(f"--- 4-Hour Pass Plan Calculation ---")
    print(f"Target Duration:        {duration_hours} hours ({total_time_sec} sec)")
    print(f"Science Time/Obs:       {sci_time_sec} sec")
    print(f"Execution Time/Obs:     {exec_time_sec} sec\n")
    
    print(f"--- APT Inputs ---")
    print(f"Observations/Segment:   {obs_per_segment}")
    print(f"Visits/Segment:         {visits_per_segment}")
    print(f"Visits/Pass:            {visits_per_pass} (assuming {num_targets} target(s) in this Pass Plan)\n")
    
    print(f"--- Actual Totals per Target ---")
    print(f"Total Execution Time:   {actual_exec_time / 3600:.2f} hours ({actual_exec_time} sec)")
    print(f"Total Science Time:     {actual_sci_time / 3600:.2f} hours ({actual_sci_time} sec)")
    print(f"Observation Efficiency: {efficiency:.1f}%")

# Run the calculation for our 4-hour block, assuming 1 target per Pass Plan for now
calculate_roman_pass_plan(duration_hours=4, sci_time_sec=98, exec_time_sec=111, num_targets=1)

--- 4-Hour Pass Plan Calculation ---
Target Duration:        4 hours (14400 sec)
Science Time/Obs:       98 sec
Execution Time/Obs:     111 sec

--- APT Inputs ---
Observations/Segment:   129
Visits/Segment:         1
Visits/Pass:            1 (assuming 1 target(s) in this Pass Plan)

--- Actual Totals per Target ---
Total Execution Time:   3.98 hours (14319 sec)
Total Science Time:     3.51 hours (12642 sec)
Observation Efficiency: 88.3%


In [6]:
exec_time = 111 # seconds
total_time_sec = 4 * 3600 # 4 hours in seconds
obs_per_segment = int(total_time_sec // exec_time) # This gives us the number of observations that can fit into a 4-hour block without exceeding it.
print(f"Observations per Segment: {obs_per_segment}")

Observations per Segment: 129


In [9]:
5.17 * 24 / 4 

31.02